In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from prophet import Prophet

# -----------------------------
# 1️⃣ 데이터 로드 및 전처리
# -----------------------------
with open("realistic_codef_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

invoices = pd.DataFrame(data["invoices"])
loans = pd.DataFrame(data["loans"])

# 날짜 컬럼 변환
invoices["issue_date"] = pd.to_datetime(invoices["issue_date"])
loans["resAccountTrDate"] = pd.to_datetime(loans["resAccountTrDate"])

# 매출/매입 합산
sales = (
    invoices[invoices["transaction_type"] == "매출"]
    .groupby("issue_date")["total_amount"]
    .sum()
)
purchases = (
    invoices[invoices["transaction_type"] == "매입"]
    .groupby("issue_date")["total_amount"]
    .sum()
)

# 대출 합산
loans["total_outflow"] = loans["resPrincipal"] + loans["resInterest"]

# 모든 날짜 합치기
all_dates = pd.date_range(invoices["issue_date"].min(), invoices["issue_date"].max())
cash_flow = pd.DataFrame({"date": all_dates})

cash_flow = cash_flow.merge(
    sales.rename("inflow"), left_on="date", right_on="issue_date", how="left"
)
cash_flow = cash_flow.merge(
    purchases.rename("purchase_outflow"),
    left_on="date",
    right_on="issue_date",
    how="left",
)

loan_outflow = loans.groupby("resAccountTrDate")["total_outflow"].sum()
cash_flow = cash_flow.merge(
    loan_outflow.rename("loan_outflow"),
    left_on="date",
    right_on="resAccountTrDate",
    how="left",
)

cash_flow.fillna(0, inplace=True)
cash_flow["net_cash_flow"] = (
    cash_flow["inflow"] - cash_flow["purchase_outflow"] - cash_flow["loan_outflow"]
)
cash_flow = cash_flow[["date", "net_cash_flow"]]

# -----------------------------
# 2️⃣ Prophet 모델 학습
# -----------------------------
df_prophet = cash_flow.rename(columns={"date": "ds", "net_cash_flow": "y"})
model = Prophet(daily_seasonality=True, interval_width=0.9)
model.fit(df_prophet)

# 이번 달 말까지 예측
end_of_month = pd.Timestamp.today().replace(day=1) + pd.offsets.MonthEnd(0)
periods = (end_of_month - cash_flow["date"].max()).days
periods = max(periods, 0)  # 음수 방지용 (예외 상황 대비)
future = model.make_future_dataframe(periods=periods)
forecast = model.predict(future)

# -----------------------------
# 3️⃣ 오늘까지 누적 현금
# -----------------------------
today = pd.Timestamp.today().normalize()
start_of_month = today.replace(day=1)

current_month = cash_flow[
    (cash_flow["date"] >= start_of_month) & (cash_flow["date"] <= today)
].copy()
current_month["cum_actual"] = current_month["net_cash_flow"].cumsum()

# -----------------------------
# 4️⃣ 이번 달 매출 예측
# -----------------------------
forecast_month = forecast[
    (forecast["ds"] >= start_of_month) & (forecast["ds"] <= end_of_month)
]

# -----------------------------
# 5️⃣ 앞으로 나갈 대출 예정 (종류별 집계)
# -----------------------------
loan_schedule = (
    loans.groupby(["resAccountTrDate", "resLoanKind"])["total_outflow"]
    .sum()
    .unstack(fill_value=0)
)

# -----------------------------
# 6️⃣ n일 뒤 부족 예상 계산
# -----------------------------
future_dates = pd.date_range(today + pd.Timedelta(days=1), end_of_month)

forecast_future = (
    forecast.set_index("ds").reindex(future_dates, method="nearest")["yhat"].fillna(0)
)

loan_daily = loans.groupby("resAccountTrDate")["total_outflow"].sum()
loan_future = loan_daily.reindex(future_dates, fill_value=0)

balance = (
    current_month["cum_actual"].iloc[-1]
    + forecast_future.cumsum()
    - loan_future.cumsum()
)
insufficient = balance[balance < 0]

if not insufficient.empty:
    first_insufficient_day = insufficient.index[0]
    shortage_amount = -int(insufficient.iloc[0])
    n_days = (first_insufficient_day - today).days
    message = f"⚠️ {n_days}일 뒤에 약 {shortage_amount:,}원 부족할 수 있습니다. 미리 준비하세요!"
else:
    message = "👍 이번 달 말까지 현금은 충분할 것으로 예상됩니다."

print(message)

# -----------------------------
# 7️⃣ 차트 시각화
# -----------------------------

# 7-1. 이번 달 누적 현금 (실제)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(current_month["date"], current_month["cum_actual"] / 1e4,
        color="green", linewidth=2)
ax.set_title("이번 달 누적 매출 (실제)", fontsize=14)
ax.set_xlabel("날짜")
ax.set_ylabel("누적 금액 (만 원)")
ax.grid(alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 7-2. 이번 달 매출 예측
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(
    forecast_month["ds"], forecast_month["yhat"] / 1e4,
    color="dodgerblue", linestyle="--", linewidth=2, label="예상 매출"
)
ax.fill_between(
    forecast_month["ds"],
    forecast_month["yhat_lower"] / 1e4,
    forecast_month["yhat_upper"] / 1e4,
    alpha=0.3,
    label="예상 범위(90%)",
)
ax.set_title("이번 달 매출 예측", fontsize=14)
ax.set_xlabel("날짜")
ax.set_ylabel("예상 금액 (만 원)")
ax.grid(alpha=0.2)
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 7-3. 앞으로 나갈 대출 예정 (종류별 스택)
# 👉 단위 맞추기 & x축을 날짜 문자열로 변환

# 단위 '만 원'
loan_schedule_plot = loan_schedule / 1e4

# x축을 "MM-DD" 문자열로
loan_schedule_plot.index = loan_schedule_plot.index.strftime("%m-%d")

fig, ax = plt.subplots(figsize=(12, 4))
loan_schedule_plot.plot(kind="bar", stacked=True, ax=ax, width=0.7)

ax.set_title("앞으로 나갈 대출 예정금액 (종류별)", fontsize=14)
ax.set_xlabel("날짜")
ax.set_ylabel("출금 금액 (만 원)")
ax.grid(alpha=0.2, axis="y")

# y축 과학적 표기(1e6 등) 비활성화
ax.ticklabel_format(style="plain", axis="y")

plt.xticks(rotation=45)
plt.legend(title="대출 종류")
plt.tight_layout()
plt.show()

import pandas as pd

file_path = '/content/sme_data.csv'
df = pd.read_csv(file_path, encoding='utf-8')  # 필요시 encoding 변경
df['시도명'] = df['시도명'].fillna('전국')
df.head()

top5 = df.sort_values("중앙값 매출채권회전율", ascending=False).head(5)
bottom5 = df.sort_values("중앙값 매출채권회전율", ascending=True).head(5)

print("=== 상위 5개 ===")
display(top5)

print("=== 하위 5개 ===")
display(bottom5)

import pandas as pd
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(df["중앙값 매출채권회전율"], bins=80)
plt.xlabel("중앙값 매출채권회전율")
plt.ylabel("빈도수")
plt.title("매출채권 회전율 분포")
plt.tight_layout()
plt.show()


subset = df[df["중앙값 매출채권회전율"] < 1]  # 0~1 사이만
plt.hist(subset["중앙값 매출채권회전율"], bins=100)
plt.show()

print(df['중앙값 매출채권회전율'].mean())


import pandas as pd

# 1) 업종별 통계 CSV 불러오기
df = pd.read_csv("/content/sme_data.csv")

# ---- 함수 정의 ----
def calculate_risk(region, sector, my_sales, my_ar_amount):
    """
    region: 시도명 (예: '경북')
    sector: 업종 대분류 (예: '도매 및 소매업')
    my_sales: 내 연매출 (예: 300_000_000)
    my_ar_amount: 내 매출채권 잔액 (예: 25_000_000)
    """

    # 2) 업계 중앙값 찾기
    cond = (df["시도명"] == region) & (df["업종 대분류"] == sector)
    row = df[cond]

    if row.empty:
        return "⚠️ 해당 지역/업종 데이터가 없습니다."

    industry_turnover = row["중앙값 매출채권회전율"].values[0]

    # 3) 내 매출채권 회전율 계산 = 매출 / 매출채권
    my_turnover = my_sales / my_ar_amount

    # 4) 위험도 진단 로직
    ratio = my_turnover / industry_turnover

    if ratio >= 1.2:
        risk = "🟢 매우 양호 (회전율이 업계 대비 매우 높음)"
    elif 0.8 <= ratio < 1.2:
        risk = "🟡 보통 (업계 평균 수준)"
    elif 0.5 <= ratio < 0.8:
        risk = "🟠 주의 (업계 대비 회수속도 느림)"
    else:
        risk = "🔴 위험 (회수 지연 → 현금흐름 악화 가능성 매우 높음)"

    # 5) 회전율 → 회수일수로 변환 (일반적 기준: 365 / 회전율)
    my_days = 365 / my_turnover
    industry_days = 365 / industry_turnover

    # ---- 출력 내용 구성 ----
    result = f"""
📌 **[{region} / {sector}] 유동성 분석 결과**

■ 내 연매출: {my_sales:,.0f}원
■ 내 매출채권: {my_ar_amount:,.0f}원

### ✅ 1) 내 매출채권 회전율
- {my_turnover:.3f} 회전
- (즉, **평균 회수기간: {my_days:.1f}일**)

### 🏭 2) 업계 중앙값 회전율
- {industry_turnover:.3f} 회전
- (업계 평균 회수기간: {industry_days:.1f}일)

### 🚨 3) 위험도 평가
{risk}

→ 업계 평균보다 회수 속도가 **{(1-ratio)*100:.1f}% 느림**
→ 현금 흐름 관리에서 개선 필요 가능성 있음.
"""
    return result


# ---- 예시 실행 ----
print(
    calculate_risk(
        region="세종",
        sector="보건업 및 사회복지 서비스업",
        my_sales=300_000_000,      # 내 연매출
        my_ar_amount=40_000_000    # 내 매출채권
    )
)


import requests
import json
import pandas as pd

# 1. 데이터 요청 정보 설정
URL = 'https://ols.semas.or.kr/ols/man/SMAN051M/search.do'

# 전체 데이터를 담을 빈 리스트 초기화
all_notices = []
# 가져올 페이지 범위 설정
start_page = 1
end_page = 3 # 1, 2, 3 페이지까지 포함

# 2. 페이지 반복문 실행
for page_num in range(start_page, end_page + 1):
    print(f"🔄 {page_num}페이지 데이터를 가져오는 중...")

    # POST 요청 데이터 (payload) 설정
    payload = {
        'bltwtrTitNm': '',   # 검색어 (빈칸)
        'pageNo': str(page_num), # 현재 반복문의 페이지 번호
        'searchStd': '1'     # 검색 기준
    }

    try:
        # HTTP POST 요청 보내기
        response = requests.post(URL, data=payload)
        response.raise_for_status() # HTTP 에러 확인

        # JSON 데이터 파싱
        json_data = response.json()

        # 'result' 키 아래에 있는 목록 데이터 추출
        notice_list = json_data.get('result', [])

        if notice_list:
            # 현재 페이지의 데이터를 전체 리스트에 추가
            all_notices.extend(notice_list)
            print(f"✅ {page_num}페이지에서 {len(notice_list)}개 항목 추가 완료.")
        else:
            print(f"⚠️ {page_num}페이지에 항목이 없습니다. (더 이상 데이터가 없거나 로딩 오류)")
            # 해당 페이지에 데이터가 없으면 반복을 종료할 수도 있지만, 여기서는 3페이지까지 강제 진행

    except requests.exceptions.RequestException as e:
        print(f"❌ {page_num}페이지 POST 요청 실패: {e}")
        break # 요청 실패 시 반복문 중단
    except json.JSONDecodeError:
        print(f"❌ {page_num}페이지 JSON 파싱 실패: 응답 내용 오류.")
        break # 파싱 실패 시 반복문 중단
    except Exception as e:
        print(f"❌ {page_num}페이지 처리 중 기타 오류 발생: {e}")
        break


# 3. 전체 데이터를 Pandas DataFrame으로 변환 및 정리
if all_notices:
    df = pd.DataFrame(all_notices)

    # 필요한 열만 선택하고 한국어 제목으로 변경하여 최종 DataFrame 생성
    df_final = df[['rnum', 'loanSeCdNm', 'bltwtrClcd', 'bltwtrTitNm', 'frstRegDt']].rename(columns={
        'rnum': '번호',
        'loanSeCdNm': '대출구분',
        'bltwtrClcd': '구분',
        'bltwtrTitNm': '제목',
        'frstRegDt': '등록일'
    })

    print(f"\n\n========================================================")
    print(f"✅ 최종 추출된 공지사항 데이터 ({len(df_final)}개 항목):")
    print("========================================================")
    print(df_final)

    # 전체 항목 수 (1페이지 정보에서 가져올 수 있지만, 현재는 수집된 개수를 출력)
    if 'pagination' in json_data:
         total_count = json_data.get('pagination', {}).get('totalCount', '확인 불가')
         print(f"\n💡 전체 공지사항 수 (시스템 기준): {total_count}건")
    print(f"💡 현재 수집된 항목 수: {len(df_final)}개")

else:
    print("\n❌ 수집된 데이터가 전혀 없습니다. 요청 URL 또는 Payload를 다시 확인해 주세요.")


# 1. 라이브러리 설치 (Colab 환경에서 처음 실행 시 필요)
!pip install konlpy jamo

# 2. 필요한 라이브러리 불러오기
import re
from konlpy.tag import Okt
from collections import Counter
import pandas as pd

# ❗️❗️❗️주의: 이전 단계에서 생성된 'df_final' DataFrame을 그대로 사용합니다.
# ❗️❗️❗️만약 Colab 세션을 다시 시작했다면, 이전 POST 요청 코드를 다시 실행해야 합니다.

# 3. 데이터 준비 (df_final이 이미 있다고 가정)
# 만약 'df_final'이 정의되지 않았다면 오류가 발생합니다.

# 4. 불용어 및 패턴 정의
# 자금의 종류를 나타내지 않는 일반적인 단어나 문자를 불용어로 정의합니다.
STOPWORDS = ['신청', '안내', '게시', '자금', '자료', '이수', '교육', '수정', '대출', '특별', '직접', '대리', '만기', '연장', '상환', '유예']
# 제목에서 제거할 패턴 정의 (예: 년도, 월, 괄호 등)
CLEAN_PATTERNS = r'\[.*?\]|\(.*?\)|[\s\.\,\:\-\'\"\'`\<\>\|\=\+]+|(\d{4}년)|(\d{1,2}월)|(\d{1,2}분기)|(\d{1,2}회)|안내자료'

# 형태소 분석기 초기화
okt = Okt()
all_keywords = []

# 5. 텍스트 클리닝 및 형태소 분석
for title in df_final['제목']:
    # 1) 불필요한 패턴 제거
    cleaned_title = re.sub(CLEAN_PATTERNS, ' ', title)
    cleaned_title = cleaned_title.strip()

    # 2) 형태소 분석 및 명사 추출
    nouns = okt.nouns(cleaned_title)

    # 3) 추출된 명사 중 불용어 제거
    final_keywords = [
        word for word in nouns
        if word not in STOPWORDS and len(word) > 1 # 1글자 단어도 제외
    ]

    all_keywords.extend(final_keywords)

# 6. 키워드 빈도 계산
word_counts = Counter(all_keywords)
top_10_keywords = word_counts.most_common(10)

# 7. 결과 출력
print("==========================================================")
print(f"💰 공고 제목 기반 TOP 10 핵심 정책자금 키워드 (총 {len(all_keywords)}개 키워드 분석)")
print("==========================================================")

# 결과를 DataFrame으로 변환하여 보기 쉽게 출력
df_keywords = pd.DataFrame(top_10_keywords, columns=['키워드', '빈도수'])
print(df_keywords)

# gensim 라이브러리 설치 (토픽 모델링을 위해 필요)
!pip install gensim

# 필요한 라이브러리 불러오기
import re
from konlpy.tag import Okt
from collections import Counter
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd

# ❗️❗️❗️주의: 이전 단계에서 생성된 'df_final' DataFrame이 정의되어 있어야 합니다.

# --- 전처리 및 키워드 추출 함수 정의 (이전 단계와 동일) ---
# 자금의 종류를 나타내지 않는 일반적인 단어나 문자를 불용어로 정의합니다.
STOPWORDS = ['신청', '안내', '게시', '자금', '자료', '이수', '교육', '수정', '대출', '특별', '직접', '대리', '만기', '연장', '상환', '유예', '접수', '소상', '공인', '정책', '지원']
CLEAN_PATTERNS = r'\[.*?\]|\(.*?\)|[\s\.\,\:\-\'\"\'`\<\>\|\=\+]+|(\d{4}년)|(\d{1,2}월)|(\d{1,2}분기)|(\d{1,2}회)|안내자료'

okt = Okt()
tokenized_docs = [] # 토큰화된 문서 리스트 (LDA 모델 입력)

for title in df_final['제목']:
    # 1) 불필요한 패턴 제거
    cleaned_title = re.sub(CLEAN_PATTERNS, ' ', title)
    cleaned_title = cleaned_title.strip()

    # 2) 형태소 분석 및 명사 추출
    nouns = okt.nouns(cleaned_title)

    # 3) 추출된 명사 중 불용어 제거 및 1글자 제외
    final_keywords = [
        word for word in nouns
        if word not in STOPWORDS and len(word) > 1
    ]

    tokenized_docs.append(final_keywords)

# 4. Dictionary 및 Corpus 생성 (LDA 모델 학습을 위한 형식)
# 단어 ID를 매핑하는 Dictionary 생성
dictionary = corpora.Dictionary(tokenized_docs)

# DTM (문서-단어 행렬) Corpus 생성: 각 문서에서 단어의 빈도수를 인코딩
corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

print(f"✅ 코퍼스 생성 완료. 총 문서 수: {len(corpus)}, 총 고유 단어 수: {len(dictionary)}")

# 토픽 개수 설정 (K=3으로 설정)
NUM_TOPICS = 3

# LDA 모델 학습
# passes: 학습 횟수 (많을수록 정확하지만 시간 소요)
lda_model = LdaModel(corpus,
                     num_topics=NUM_TOPICS,
                     id2word=dictionary,
                     passes=15,
                     random_state=42)

# 4. 결과 출력
print("\n========================================================")
print(f"🔬 LDA 토픽 분석 결과 ({NUM_TOPICS}개 토픽)")
print("========================================================")

topics_data = []

# 토픽별 주요 키워드 추출 및 출력
for idx, topic in lda_model.show_topics(num_words=5, formatted=False):
    topic_keywords = ", ".join([word for word, prob in topic])
    topics_data.append({
        '토픽 ID': idx,
        '주요 키워드': topic_keywords
    })

# DataFrame으로 정리하여 출력
df_topics = pd.DataFrame(topics_data)
print(df_topics)

# 공고문별 주 토픽을 찾아 df_final에 추가
def get_dominant_topic(lda_model, corpus, df):
    # 문서별 토픽 분포 추출
    topic_distribution = [lda_model.get_document_topics(bow) for bow in corpus]

    # 주 토픽 ID와 확률을 저장할 리스트
    topic_results = []

    for doc_topics in topic_distribution:
        # 확률이 가장 높은 토픽을 찾습니다.
        dominant_topic = max(doc_topics, key=lambda x: x[1])
        topic_results.append({
            'Dominant_Topic_ID': dominant_topic[0],
            'Topic_Probability': dominant_topic[1]
        })

    df_topics_info = pd.DataFrame(topic_results)

    # 기존 df_final에 주 토픽 정보 병합
    df = pd.concat([df.reset_index(drop=True), df_topics_info], axis=1)

    return df

# df_final에 주 토픽 정보 추가 (df_topics_labeled 변수에 저장)
df_topics_labeled = get_dominant_topic(lda_model, corpus, df_final)

print("✅ 공고문별 주 토픽 라벨링 완료 (새로운 칼럼 'Dominant_Topic_ID' 추가)")
print(df_topics_labeled[['제목', 'Dominant_Topic_ID', 'Topic_Probability']].head())

def recommend_funding(user_query, df_labeled, lda_model, dictionary, num_recommendations=5):

    # 1. 사용자 질문 전처리 (공고문과 동일한 방식으로)
    cleaned_query = re.sub(CLEAN_PATTERNS, ' ', user_query).strip()
    query_nouns = [word for word in okt.nouns(cleaned_query)
                   if word not in STOPWORDS and len(word) > 1]

    if not query_nouns:
        return "❌ 추천 실패: 질문에서 유효한 키워드를 추출할 수 없습니다. 더 구체적인 설명을 해주세요."

    # 2. 질문을 BoW 벡터로 변환
    query_bow = dictionary.doc2bow(query_nouns)

    # 3. LDA 모델을 사용하여 질문의 토픽 분포 예측
    query_topics = lda_model.get_document_topics(query_bow)

    # 4. 가장 확률이 높은 주 토픽 ID 추출
    dominant_topic_id = max(query_topics, key=lambda x: x[1])[0]
    dominant_topic_prob = max(query_topics, key=lambda x: x[1])[1]

    print(f"\n🧠 사용자 질문의 분석 결과:")
    print(f"   추출된 키워드: {query_nouns}")
    print(f"   가장 높은 토픽 ID: {dominant_topic_id} (확률: {dominant_topic_prob:.2f})")

    # 5. 해당 토픽에 속하는 공고문 필터링 및 추천
    recommendations = df_labeled[
        df_labeled['Dominant_Topic_ID'] == dominant_topic_id
    ].sort_values(by='등록일', ascending=False).head(num_recommendations) # 최신순 5개 추천

    return recommendations[['번호', '대출구분', '제목', '등록일', 'Dominant_Topic_ID']]

# ----------------------------------------------------
# 3. 예시 사용
# ----------------------------------------------------

# 예시 1: 경영이 어려워서 도움을 받고 싶은 경우 (일반 경영 안정 토픽 예상)
query_1 = "요즘 매출이 너무 안 나와서 일시적으로 자금 운영에 어려움을 겪고 있습니다."
recommendation_1 = recommend_funding(query_1, df_topics_labeled, lda_model, dictionary, num_recommendations=5)

print("\n\n========================================================")
print("✅ 사용자 맞춤형 추천 결과 (예시 1: 경영 애로)")
print("========================================================")
print(recommendation_1)

# 예시 2: 새로운 기술로 사업을 키우고 싶은 경우 (성장/혁신 토픽 예상)
query_2 = "새로운 제품 개발을 위해 연구개발 자금이 필요하고 혁신적으로 성장하고 싶어요."
recommendation_2 = recommend_funding(query_2, df_topics_labeled, lda_model, dictionary, num_recommendations=3)

print("\n\n========================================================")
print("✅ 사용자 맞춤형 추천 결과 (예시 2: 혁신 성장)")
print("========================================================")
print(recommendation_2)